# Smart Waste Segregation - YOLO detector training on Colab (TACO)

Trains **yolo11n** (COCO-pretrained) on the two TACO datasets made by `src/taco_to_yolo.py`, one
after the other, then scores each on **val and test**:

1. `taco_yolo_6class` - boxes labelled cardboard / glass / metal / paper / plastic / trash
2. `taco_yolo_1class` - the same boxes, all labelled `waste` (the localiser for the two-stage system)

Both use the same image-level split (`splits/taco_*.txt`), so their numbers are comparable.

**Before you run this** (see `docs/COLAB.md`, section 5):
1. `src/train_yolo.py`, `src/evaluate_yolo.py` and `splits/taco_*.txt` are committed and pushed.
2. `data/processed/taco_yolo.zip` (made by `python src/taco_to_yolo.py --zip`, ~1 GB) is on Drive.
3. **Runtime -> Change runtime type -> T4 GPU.**

**Resumable:** training writes its checkpoints straight to Drive, so after a disconnect re-run all
cells: a finished model is skipped and an unfinished one resumes from its last epoch.

## 0. Configuration

In [ ]:
# ---- your repo ---------------------------------------------------------------
REPO_URL  = "https://github.com/aarya5p/smart-waste-management.git"
BRANCH    = "main"
REPO_IS_PRIVATE = False

# ---- Drive ---------------------------------------------------------------------
DRIVE_TACO_ZIP   = "/content/drive/MyDrive/minor_project/taco_yolo.zip"
DRIVE_YOLO_DIR   = "/content/drive/MyDrive/minor_project/yolo"     # training runs + saved results
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/minor_project/outputs"  # result zips

# ---- local (fast) paths on the Colab VM ---------------------------------------
WORK_DIR     = "/content"
DATASETS_DIR = "/content/datasets"

# ---- training settings (same for both datasets) -------------------------------
MODEL    = "yolo11n.pt"   # COCO-pretrained nano model
IMGSZ    = 640
EPOCHS   = 100
PATIENCE = 20             # early stopping on VALIDATION fitness
SEED     = 42
BATCH    = 16
WORKERS  = 2
ULTRALYTICS_VERSION = "8.4.165"   # same as requirements.txt

# Trained in this order. run_name = <model>_<dataset tag>_seed<seed>
RUNS = [
    {"dataset": "taco_yolo_6class", "run_name": f"yolo11n_taco6_seed{SEED}"},
    {"dataset": "taco_yolo_1class", "run_name": f"yolo11n_taco1_seed{SEED}"},
]
EVAL_SPLITS = ["val", "test"]   # val = decisions, test = report only (rule 2)

import os
from pathlib import Path
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
TRAIN_DIR = Path(DRIVE_YOLO_DIR, "train")    # Ultralytics run folders (weights/last.pt live here)
STORE_DIR = Path(DRIVE_YOLO_DIR, "store")    # per-run results + LOG rows + DONE marker

In [ ]:
import subprocess, sys, time

def run(cmd, cwd=None, check=True):
    """Run a command and stream its output live."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode} after {time.time()-t0:.0f}s")
    print(f"[done in {time.time()-t0:.0f}s]\n", flush=True)
    return p.returncode

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## 2. Clone the repo and install Ultralytics

In [ ]:
import shutil

clone_url = REPO_URL
if REPO_IS_PRIVATE:
    from getpass import getpass
    token = getpass("GitHub personal access token (not stored, not printed): ")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")

if Path(REPO_DIR, ".git").exists():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, clone_url, REPO_DIR], cwd=WORK_DIR)
del clone_url
run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR)

required = ["src/train_yolo.py", "src/evaluate_yolo.py", "src/train_classifier.py",
            "splits/taco_train.txt", "splits/taco_val.txt", "splits/taco_test.txt"]
missing = [f for f in required if not Path(REPO_DIR, f).exists()]
if missing:
    raise SystemExit("Missing from the clone (commit and push them first):\n  " + "\n  ".join(missing))
print("All required files present.")

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
Path(REPO_DIR, "pip_freeze_colab_yolo.txt").write_text(freeze)
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. Copy the TACO datasets from Drive to local disk

`taco_yolo.zip` holds the 6-class dataset complete plus the 1-class labels; the 1-class images are
the same files, so they are recreated here with hard links (`cp -al`, no extra disk space).
The cell then checks that every image matches the committed split lists and writes a `data.yaml`
with this machine's paths.

In [ ]:
import zipfile

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]   # same order as the classifiers
d6, d1 = Path(DATASETS_DIR, "taco_yolo_6class"), Path(DATASETS_DIR, "taco_yolo_1class")

if not Path(DRIVE_TACO_ZIP).exists():
    raise SystemExit(f"Not found on Drive: {DRIVE_TACO_ZIP}\nUpload data/processed/taco_yolo.zip there.")
if not (d6 / "images" / "train").is_dir():
    t0 = time.time()
    Path(DATASETS_DIR).mkdir(parents=True, exist_ok=True)
    local_zip = Path(WORK_DIR, "taco_yolo.zip")
    shutil.copy2(DRIVE_TACO_ZIP, local_zip)            # one big file over Drive is fast
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(DATASETS_DIR)
    local_zip.unlink()
    print(f"Unzipped in {time.time()-t0:.0f}s")
if not (d1 / "images").exists():
    run(["cp", "-al", str(d6 / "images"), str(d1 / "images")])

# ---- verify against the committed split lists (rule 1: the split is fixed)
def out_name(fn):                      # same rule as src/taco_to_yolo.py
    return fn.replace("/", "_").rsplit(".", 1)[0] + ".jpg"

for split in ["train", "val", "test"]:
    want = {out_name(l) for l in Path(REPO_DIR, "splits", f"taco_{split}.txt").read_text().split()}
    for d in (d6, d1):
        have = {p.name for p in (d / "images" / split).glob("*.jpg")}
        labels = {p.stem + ".jpg" for p in (d / "labels" / split).glob("*.txt")}
        if have != want or labels != want:
            raise SystemExit(f"{d.name}/{split}: images/labels do not match splits/taco_{split}.txt")
    print(f"{split:5s} {len(want):5d} images - match splits/taco_{split}.txt in both datasets")

# ---- data.yaml with absolute paths on this VM
for d, names in ((d6, CLASSES), (d1, ["waste"])):
    (d / "data.yaml").write_text(
        f"path: {d}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\n"
        "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names)))
print("data.yaml written for both datasets")

## 4. Train and evaluate (resumable)

For each dataset, in order:

1. **Skip** if `DRIVE_YOLO_DIR/store/<run_name>/DONE` exists (finished in an earlier session).
2. **Train** with `src/train_yolo.py`, writing straight to `DRIVE_YOLO_DIR/train/<run_name>/`.
   If that folder has `weights/last.pt` from an interrupted session, training **resumes** from it;
   if it has `train_done.json`, training is skipped. Early stopping and `best.pt` use **val** only.
3. **Evaluate** `best.pt` with `src/evaluate_yolo.py` on **val and test** -> `results/<run_name>/val|test/`
   (mAP@0.5, mAP@0.5:0.95, per-class AP).
4. **Save** `results/<run_name>/` and the run's LOG.md rows to `DRIVE_YOLO_DIR/store/<run_name>/`
   (copied to `.partial`, renamed, `DONE` written last).

> **Rule 2.** Decisions (e.g. which detector goes into the two-stage system) come from the **val**
> numbers only. Test numbers are for the final tables.

In [ ]:
import json
from datetime import datetime

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
STORE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = Path(REPO_DIR, "LOG.md")


def n_log_lines():
    return len(LOG_PATH.read_text().splitlines()) if LOG_PATH.exists() else 0


def new_log_rows(since):
    return [l for l in LOG_PATH.read_text().splitlines(keepends=True)[since:] if l.startswith("|")]


def save_to_drive(rn, log_rows):
    tmp, final = STORE_DIR / f"{rn}.partial", STORE_DIR / rn
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(Path(REPO_DIR, "results", rn), tmp / "results" / rn)
    (tmp / "log_rows.md").write_text("".join(log_rows))
    (tmp / "DONE").write_text(datetime.now().isoformat())
    if final.exists():
        old = STORE_DIR / "_superseded" / f"{rn}_{datetime.now():%Y%m%d_%H%M%S}"
        old.parent.mkdir(exist_ok=True)
        shutil.move(str(final), str(old))
    tmp.rename(final)


summary, failures = [], []
for i, r in enumerate(RUNS, 1):
    rn, data_yaml = r["run_name"], str(Path(DATASETS_DIR, r["dataset"], "data.yaml"))
    print("=" * 78 + f"\n[{i}/{len(RUNS)}] {rn}  ({r['dataset']})\n" + "=" * 78, flush=True)
    if (STORE_DIR / rn / "DONE").exists():
        print("already finished and saved on Drive - SKIPPED\n")
        continue
    try:
        run([sys.executable, "src/train_yolo.py", "--data", data_yaml, "--run_name", rn,
             "--project", TRAIN_DIR, "--model", MODEL, "--imgsz", IMGSZ, "--epochs", EPOCHS,
             "--patience", PATIENCE, "--batch", BATCH, "--seed", SEED, "--workers", WORKERS],
            cwd=REPO_DIR)
        train_row_file = TRAIN_DIR / rn / "log_row_train.md"
        rows = train_row_file.read_text().splitlines(keepends=True) if train_row_file.exists() else []

        shutil.rmtree(Path(REPO_DIR, "results", rn), ignore_errors=True)
        before = n_log_lines()
        row = {"run": rn}
        for split in EVAL_SPLITS:
            run([sys.executable, "src/evaluate_yolo.py", "--weights", TRAIN_DIR / rn / "weights" / "best.pt",
                 "--data", data_yaml, "--split", split, "--run_name", rn,
                 "--imgsz", IMGSZ, "--batch", BATCH], cwd=REPO_DIR)
            m = json.loads(Path(REPO_DIR, "results", rn, split, "metrics.json").read_text())
            row[f"{split}_mAP50"], row[f"{split}_mAP50-95"] = m["map50"], m["map50_95"]
        rows += new_log_rows(before)
        save_to_drive(rn, rows)
        print(f"Saved {rn} to Drive ({len(rows)} LOG rows)\n", flush=True)
        summary.append(row)
    except Exception as e:      # one failure must not lose the other run; re-run the cell to retry
        print(f"!!! {rn} FAILED: {e}\n", flush=True)
        failures.append(rn)

print("Failed (re-run this cell to retry):", failures or "none")
if summary:
    import pandas as pd
    print(pd.DataFrame(summary).to_string(index=False))
    print("\nDecide from the val_* columns ONLY; test_* is for the final report tables.")

## 5. Bundle everything on Drive into one zip

Collects every finished run from Drive (this session and earlier ones) into
`waste_yolo_results_<timestamp>.zip` in `DRIVE_OUTPUT_DIR`:
`runs/yolo/<run>/` (weights, results.csv, training plots, args.yaml), `results/<run>/val|test/`,
`LOG_rows.md` and `pip_freeze_colab_yolo.txt`.

In [ ]:
import zipfile
from datetime import datetime

zip_path = Path(WORK_DIR, f"waste_yolo_results_{datetime.now():%Y%m%d_%H%M%S}.zip")
finished = sorted(d for d in STORE_DIR.iterdir()
                  if d.is_dir() and (d / "DONE").exists() and not d.name.startswith("_"))
print(f"{len(finished)} finished run(s):", [d.name for d in finished])

log_rows = []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in finished:
        for p in (d / "results").rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(d))
        for p in (TRAIN_DIR / d.name).rglob("*"):
            if p.is_file():
                z.write(p, Path("runs/yolo") / p.relative_to(TRAIN_DIR))
        log_rows += (d / "log_rows.md").read_text().splitlines(keepends=True)
    z.writestr("LOG_rows.md", "".join(log_rows))
    fr = Path(REPO_DIR, "pip_freeze_colab_yolo.txt")
    if fr.exists():
        z.write(fr, fr.name)

Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
dest = Path(DRIVE_OUTPUT_DIR, zip_path.name)
shutil.copy2(zip_path, dest)
print(f"{zip_path.stat().st_size/1024**2:.1f} MB -> {dest}")

## After the run

1. Download the zip; unzip `runs/yolo/` and `results/` into the repo.
2. Append `LOG_rows.md` to `LOG.md` (drop exact duplicates only). Per run: 1 train row + 2 eval rows.
3. Commit `results/` (metrics.json, PR curves, confusion matrices). `runs/` stays untracked.

**Latency caveat (rule 8):** speed numbers in `metrics.json` are Colab GPU batch timings, not Pi latency.